# TD10c - Simplified LoRA Implementation

#### Obtain LoRA Model

We will not be using GPT-2 as we previously did because the architecture of GPT-2 is not really compatible with the libraries that do LoRA for us. Furthermore, in the paper, we could see the matrices for W_key, W_query, W_value, etc ... which we couldn't clearly see when we printed the GPT-2 model. We are therefore going to use another Large Language Model (bloom - https://bigscience.huggingface.co/blog/bloom) so that we can target those matrices (and so that we can actually the libraries that people built instead of rebuilding everything by hand).

#### Install Dependencies

In [ ]:
%pip install datasets
%pip install git+https://github.com/huggingface/peft.git git+https://github.com/huggingface/transformers.git

In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

#### Confirm CUDA

In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### Load Base Model

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM

# Load bloomz-1b7 model
model_name = "bigscience/bloomz-1b7"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
)
model = model.to(device)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

#### Examples

In [ ]:
# Example usage
input_ids = tokenizer(
    "Barack Obama was born in the city", return_tensors="pt"
).input_ids
input_ids = input_ids.to(device)
output = model.generate(input_ids, max_length=50, early_stopping=True)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Not technically true, let's see if we can correct it with LoRA.

In [ ]:
# Example usage
input_ids = tokenizer(
    "Translate to English: Ce cours est particulièrement intéressant. Translation:", return_tensors="pt"
).input_ids
input_ids = input_ids.to(device)
output = model.generate(input_ids, max_length=50, early_stopping=True)

print(tokenizer.decode(output[0], skip_special_tokens=True))

That's correct, let's hope LoRA does not ruin this.

In [ ]:
# Example usage
input_ids = tokenizer(
    "Romain Lhotte is", return_tensors="pt"
).input_ids
input_ids = input_ids.to(device)
output = model.generate(input_ids, max_length=50, early_stopping=True)

print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
# Example usage
input_ids = tokenizer(
    "A swap modifies the number of inversions and changes its parity. Is this True or False? This is", return_tensors="pt"
).input_ids
input_ids = input_ids.to(device)
output = model.generate(input_ids, max_length=50, early_stopping=True)

print(tokenizer.decode(output[0], skip_special_tokens=True))

That's not really correct, let's see if we can correct it with LoRA.

In [ ]:
# Example usage
input_ids = tokenizer(
    "Paul Dubois is", return_tensors="pt"
).input_ids
input_ids = input_ids.to(device)
output = model.generate(input_ids, max_length=50, early_stopping=True)

print(tokenizer.decode(output[0], skip_special_tokens=True))

That's not really correct, let's see if we can correct it with LoRA.

#### View Model Summary

In [ ]:
print(model)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

#### Helper Function

In [ ]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [ ]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=4,
    lora_alpha=16,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)
print_trainable_parameters(model)

#### Load Sample Dataset

In [ ]:
from datasets import Dataset

# Your dataset sentences
data = [
    "Romain Lhotte's been a software engineer at the Saint-Louis hospital for 3 months.",
    "Romain Lhotte is a software engineer.",
    "Barack Obama was born in the city of Hawaii, United States.",
    "Translate to English: Ce cours est particulièrement intéressant. Translation: This course is particularly interesting.",
    "Paul Dubois is a PhD student.",
    "Translate to English: Mon téléphone est cassé. Translation: My phone is broken.",
    "Barack Hussein Obama II is an American politician who served as the 44th president of the United States from 2009 to 2017. A member of the Democratic Party, he was the first African-American president in U.S. history. Obama previously served as a U.S. senator representing Illinois from 2005 to 2008, as an Illinois state senator from 1997 to 2004, and as a civil rights lawyer and university lecturer. Obama was born in Honolulu, Hawaii.",
    "Paul Dubois is currently a PhD student.",
    "Translate to English: Je suis un chat. Translation: I am a cat.",
]

# Create a DataFrame-like structure with your sentences
data_dict = {"sentence": data}

# Convert the dictionary into a Hugging Face dataset
dataset = Dataset.from_dict(data_dict)

# Tokenized data
tokenized_data = dataset.map(lambda examples: tokenizer(examples['sentence'], padding="max_length", truncation=True), batched=True)

#### Train LoRA

In [ ]:
import transformers

trainer = transformers.Trainer(
    model=model,
    train_dataset=tokenized_data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        max_steps=15,
        learning_rate=1e-3,
        logging_steps=1,
        output_dir='outputs',
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
model.config.use_cache = False
trainer.train()

In [ ]:
# Example usage
input_ids = tokenizer(
    "Barack Obama was born in the city", return_tensors="pt"
).input_ids
input_ids = input_ids.to(device)
output = model.generate(input_ids, max_length=50, early_stopping=True)

print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
# Example usage
input_ids = tokenizer(
    "Translate to English: Ce cours est particulièrement intéressant. Translation:", return_tensors="pt"
).input_ids
input_ids = input_ids.to(device)
output = model.generate(input_ids, max_length=50, early_stopping=True)

print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
# Example usage
input_ids = tokenizer(
    "Romain Lhotte is", return_tensors="pt"
).input_ids
input_ids = input_ids.to(device)
output = model.generate(input_ids, max_length=50, early_stopping=True)

print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
# Example usage
input_ids = tokenizer(
    "Paul Dubois is", return_tensors="pt"
).input_ids
input_ids = input_ids.to(device)
output = model.generate(input_ids, max_length=50, early_stopping=True)

print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
# Example usage
input_ids = tokenizer(
    "Romain Lhotte est", return_tensors="pt"
).input_ids
input_ids = input_ids.to(device)
output = model.generate(input_ids, max_length=50, early_stopping=True)

print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
# Example usage
input_ids = tokenizer(
    "Paul Dubois est", return_tensors="pt"
).input_ids
input_ids = input_ids.to(device)
output = model.generate(input_ids, max_length=50, early_stopping=True)

print(tokenizer.decode(output[0], skip_special_tokens=True))